# ECR research workbench
Run cells from top to bottom. Edit the configuration cell and rerun from **Run backtest** to explore; the session loads inputs once and caches up to eight score panels. Recreate the session when input files change.

We retain the exact workbook formula:
`(w_DBB × DBB + w_Reset × Reset) × Buffer^γ_Buffer × max(0,Cap)^γ_Cap + period_bonus × Period`.
The standard multiplicative ECR selects funds. The conservative experiment varies only the buffer-integrity exit floor; there are no three-component allocation controls. Higher gate exponents penalize depleted integrity more strongly; they are not portfolio weights. A larger DBB coefficient is not automatically a more defensive investment.

Default settings reproduce the workbook. Twelve cohorts share overlapping history and are not twelve independent trials. Results are exploratory, not a parameter-selection holdout.

In [ ]:
from pathlib import Path
from dataclasses import replace, asdict
from datetime import datetime, timezone
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config/ecr.json").exists())
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"
from config.settings import ScoreConfig, BacktestConfig
from backtesting.research import ResearchSession, initial_experiments, export_comparison
from visualization.research_plots import dashboard, comparison_figures
from utils.exporters import export_results

## Load once

In [ ]:
session = ResearchSession(ROOT / "input_data/website_metrics.csv",
                          ROOT / "input_data/benchmark_ts.csv", ROOT / "input_data/roll_dates.csv")
print(f"Loaded {len(session.metrics):,} observations.")

## Configuration — standard ECR selection and buffer-integrity trigger
The default conservative run triggers when `buffer_integrity < 0.80`, relative to an **ideal fresh score of 1.00**, then selects the highest ECR. Adjust `metric_floor` to test other fractions. The regular ECR threshold is retained as the comparison baseline.

`buffer_drop` is a separate, older purchase-relative mode and is not used in this experiment. All triggers respect the review schedule and minimum holding period. `execution_lag=0` is the theoretical same-close convention; use `1` for next-close execution. The sensitivity section also applies 5 bp per switch.

In [ ]:
score = ScoreConfig(
    dbb_weight=1.5, reset_weight=1.2,
    buffer_exponent=0.5, cap_exponent=0.3, period_bonus=0.3,
    sigma_down=0.07, sigma_up=0.05, period_center=180., period_width=75.,
)
backtest = BacktestConfig(
    threshold=1.5, comparator="<", review="monthly", min_holding_days=0,
    initial_selection="highest_score", trigger="metric_floor", buffer_metric="buffer_integrity", buffer_drop=0.20, metric_floor=0.80,
    execution_lag=0, cost_bps=0., start_date="2021-01-19", end_date=None,
    candidate_audit=True,
)
COHORT = "Jan"  # Choose a cohort for the charts below.

## Run backtest

In [ ]:
result = session.run(score, backtest)
display(result["summary"])
print(result["timings"], "Score panels built:", session.score_builds)
print("Missing benchmark sessions remain NaN; see the coverage column.")

## Interactive NAV, drawdowns, and buffer deterioration
Hover to inspect dates, click legend items to hide series, and drag to zoom. Entry/switch markers identify the selected fund.

In [ ]:
nav_figure = dashboard(result, COHORT)
nav_figure.show()

## Why did we trade?
Review decisions are distinct from executed switches. The daily buffer chart reflects the fund held at close; the decision table shows the outgoing fund and both trigger conditions.

In [ ]:
display(result["diagnostics"].query("Cohort == @COHORT"))
display(result["decisions"].query("Cohort == @COHORT").tail(20))
# Uncomment to inspect the entire fund ranking on a particular review date:
# display(result["candidates"].query("Cohort == @COHORT and Date == '2025-01-17'"))

## Controlled buffer-integrity experiments
The comparison keeps the same multiplicative ECR for selection in every run. It compares an ECR-below-1.5 trigger with buffer-integrity floors of **0.70, 0.80, and 0.90**. All use strict `<`, highest-ECR initial selection, and identical dates, costs, execution timing, review cadence and holding periods.

Fresh buffer integrity is defined as **1.00**. A floor of 0.80 means less than 80% of ideal fresh protection, not a 20% decline from the purchase-date score. Replacement selection always chooses the highest ECR, not the highest buffer integrity. No portfolio allocations or score weights are swept.

In [ ]:
experiments = initial_experiments(score, backtest)
results, comparison = session.compare(experiments)
display(comparison)
print("Cached score panels:", session.score_builds)

In [ ]:
heatmap, tradeoff = comparison_figures(results, comparison)
heatmap.show()
tradeoff.show()

## Execution and cost sensitivity
Repeat the same experiments with next-close execution and a 5 bp cost per switch. This is sensitivity analysis on the same historical sample, not out-of-sample validation.

In [ ]:
sensitive = [replace(e, backtest=replace(e.backtest, execution_lag=1, cost_bps=5.)) for e in experiments]
sensitivity_results, sensitivity_table = session.compare(sensitive)
display(sensitivity_table)
display((sensitivity_table[["Mean vs BUFR", "Worst Drawdown"]] - comparison[["Mean vs BUFR", "Worst Drawdown"]]).rename(columns=lambda c: c + " change"))

## Export explicitly
Set `SAVE = True` to save the current run, exact configuration, comparison tables, and self-contained interactive HTML. Each export creates a new directory.

In [ ]:
SAVE = False
if SAVE:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    output = ROOT / "output" / ("notebook-" + stamp)
    manifest = {**result["parameters"], "sources": session.sources, "timings": result["timings"]}
    export_results(result, output, manifest, plots=False)
    (output / "config.json").write_text(json.dumps({"score": asdict(score), "backtest": asdict(backtest)}, indent=2))
    export_comparison(results, output / "experiments")
    export_comparison(sensitivity_results, output / "sensitivity")
    for name, figure in [("dashboard", nav_figure), ("cohort_heatmap", heatmap), ("tradeoff", tradeoff)]:
        figure.write_html(output / (name + ".html"), include_plotlyjs=True)
    print(output)

## Visualization roadmap
- **Implemented:** linked NAV/drawdown/buffer panels with switch markers; cohort excess-return heatmap; return/drawdown/turnover comparison; event and candidate tables.
- **Next:** component contributions on each selection date to explain what changed the ranking. For the multiplicative formula, distinguish the DBB/reset contributions after gating from the additive period bonus.
- **Next:** holdings timeline and fund-rank heatmap across the 12-fund universe to expose concentration and repeated switching.
- **Next:** two-dimensional threshold/gate sensitivity surfaces with a separate validation period, highlighting broad stable regions rather than the single best cell.
- **Next:** paired per-cohort differences against the workbook baseline and rolling downside performance during drawdowns.

Interpretation matters: a larger DBB weight or stricter buffer gate is a hypothesis about defense, not proof of lower realized drawdown.